# Clean PSA underemployment data
Reads `data/psa_underemployment_2021_2026_thousands.csv` (PSA Labor Force Survey, thousands of persons), drops the `Annual` summary rows and writes the tidy JSON the dashboard loads. Put the JSON in `web/data/`.

In [ ]:
import pandas as pd

PATH = "../data/psa_underemployment_2021_2026_thousands.csv"   # in Colab: upload the csv and change this path
MONTHS = ["January","February","March","April","May","June","July","August","September","October","November","December"]

raw = pd.read_csv(PATH)
raw = raw[raw["Month"] != "Annual"]            # annual summary rows are not monthly observations
assert len(raw) == 68, len(raw)

long = raw.melt(id_vars=["Year", "Month"], var_name="series", value_name="thousands")
parts = long["series"].str.extract(r"^(Both Sexes|Male|Female) - (\d+) - (\d+) Years Old$")
long["sex"] = parts[0].replace({"Both Sexes": "Both sexes"})
long["age"] = parts[1] + "-" + parts[2]
long["month_num"] = long["Month"].map({m: i + 1 for i, m in enumerate(MONTHS)})
tidy = long.rename(columns={"Year": "year", "Month": "month"})[["year", "month", "month_num", "sex", "age", "thousands"]]
tidy = tidy.sort_values(["year", "month_num", "sex", "age"]).reset_index(drop=True)
print(tidy.shape, tidy.isna().sum().sum(), "missing")
tidy.to_json("underemployment_tidy.json", orient="records", indent=1)
tidy.head()